In [ ]:
import pandas as pd
import logging
from marginal_emissions.utils.helper import search_df, check_encoding
logging.basicConfig(level=logging.INFO, format='[%(levelname)s][%(asctime)s][%(filename)s]_%(message)s')

In [ ]:
kws_path = '../data/raw/bundesnetzagentur/kraftwerksliste.csv'
cdc_path = '../data/raw/bundesnetzagentur/ZuUndRueckbau.xlsx' # cdc = commission/decommission
wind_path = '~/Downloads/stundenwerte_FF_00282_19710801_20241231_hist/produkt_ff_stunde_19710801_20241231_00282.txt' # avgeraged by hour
wind_path2 = '~/Downloads/stundenwerte_F_00282_19490101_20241231_hist/produkt_f_stunde_19490101_20241231_00282.txt' # hourly values
ele_em = '../data/raw/electricitymaps/DE_2024_hourly.csv'

In [ ]:
kws_encoding = check_encoding(path=kws_path)

# To check exact column names
kws_cols = pd.read_csv(kws_path, sep=';', skiprows=11, header=0, nrows=5, encoding='Windows-1252')

if kws_encoding:
    kws = pd.read_csv(kws_path, sep=';', skiprows=11, header=0, encoding=kws_encoding, usecols=[
        'Anzeige-Name der Stromerzeugungseinheit',
        'Anlagenbetreiber',
        'Anschlussnetzbetreiber',
        'Bundesland der Einheit',
        'Ort der Einheit',
        'Kraftwerksstatus der Einheit',
        'Energieträger',
        'Speichertechnologie',
        'Wärmeauskopplung (KWK)\n(ja/nein)',
        'Nettonennleistung (elektrische Wirkleistung) in MW',
        'Technologie der Stromerzeugung',
        'Spannungsebene'
    ]).rename(columns={
        'Anzeige-Name der Stromerzeugungseinheit': 'Name',
        'Bundesland der Einheit': 'Bundesland',
        'Ort der Einheit': 'Ort',
        'Kraftwerksstatus der Einheit': 'Status',
        'Wärmeauskopplung (KWK)\n(ja/nein)': 'KWK',
        'Nettonennleistung (elektrische Wirkleistung) in MW': 'Net_Wirkleistung',
        'Technologie der Stromerzeugung': 'Technologie'
    })

    kw_carrier = kws[kws['Energieträger'].str.contains('Speicher', case=False, na=False)].drop(columns='Technologie')
    kw_carrier_pump = kw_carrier[kw_carrier['Speichertechnologie'] == 'Pumpspeicher']
    kw_solar = search_df(kws, 'Solar').copy() # use .copy() to avoid modifying
    kw_solar.Net_Wirkleistung = kw_solar.Net_Wirkleistung.str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float) # makes float from string
    kw_wind = kws[kws['Energieträger'].str.contains('wind', case=False, na=False)].copy()
    kw_wind.Net_Wirkleistung = kw_wind.Net_Wirkleistung.str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float) # makes float from string
    kw_kwk = kws[kws['KWK'] == 'Ja']

    print("# Kraftwerke: ", len(kws))
    print("# Speicher: ", len(kw_carrier_pump))
    print("# Must-Run: ", len(kw_kwk))

    # Einzeiler für eine ganze Spalte
    # df['column_name'] = df['column_name'].str.replace('.', '', regex=False).str.replace(',', '.').astype(float)

In [ ]:
kws['Anschlussnetzbetreiber'].unique()


In [ ]:
kws_cols.columns

In [ ]:
net_solar = kw_solar.Net_Wirkleistung.sum()
net_wind = kw_wind.Net_Wirkleistung.sum()

print(f"Windeinspeisung: {net_wind} MWh; Solareinspeisung: {net_solar} MWh")

In [ ]:
# Wetterdaten
import csv
wind_bam = pd.read_csv(wind_path, delimiter=';').rename(columns={
        'STATIONS_ID': 'id',
        'MESS_DATUM': 'timestamp',
        'QN_3': 'quality_lvl',
        '   F': 'speed',
        '   D': 'direction'
    }).drop(columns=['eor'])
wind_bam.timestamp = pd.to_datetime(wind_bam.timestamp, format='%Y%m%d%H')
wind_bam = wind_bam[wind_bam.timestamp.dt.year >= 2000]
wind_bam.set_index('timestamp')

wind_bam2 = pd.read_csv(wind_path2, delimiter=';').rename(columns={
    'STATIONS_ID': 'id',
    'MESS_DATUM': 'timestamp',
    'QN_8': 'quality_lvl',
    '  FF': 'speed',
    '  DD': 'direction'
}).drop(columns=['eor'])
wind_bam2.timestamp = pd.to_datetime(wind_bam2.timSonstige Konventionelle [MWh] Originalauflösungenestamp, format='%Y%m%d%H')
wind_bam2 = wind_bam2[wind_bam2.timestamp.dt.year >= 2000]
wind_bam2.set_index('timestamp')

In [ ]:
wind_bam[wind_bam.speed == wind_bam.speed.max()]

In [ ]:
wind_bam2[(wind_bam2.speed == wind_bam2.speed.max()) & (wind_bam2.quality_lvl > 0)]

In [ ]:
wind_bam2[(wind_bam2.quality_lvl > 2) & (wind_bam2.speed > 10)]

In [ ]:
max_speed

In [ ]:
# Emissions data electricitymaps
df_emissions = pd.read_csv(ele_em)

In [ ]:
df_emissions.head()

In [ ]:
df_emissions['Carbon intensity gCO₂eq/kWh (direct)'].min()